# Strava Leistungsanalyse aus einer FIT-Datei

Dieses Notebook ist für **Visual Studio Code + Jupyter** ausgelegt und analysiert eine `.fit`-Datei aus `data/raw/`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / 'src'))

from fit_loader import find_first_fit_file, load_fit_records, load_fit_session_summary
from analysis import add_time_bins, summarize_activity, rolling_best_efforts, heart_rate_zones, correlation_matrix

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
pd.set_option('display.max_columns', 50)


In [ ]:
fit_file = find_first_fit_file(PROJECT_ROOT / 'data' / 'raw')
fit_file


In [ ]:
df = load_fit_records(fit_file)
session_summary = load_fit_session_summary(fit_file)
df = add_time_bins(df)

print(f'Records geladen: {len(df):,}')
df.head()


In [ ]:
summary_df = summarize_activity(df, session_summary)
summary_df


## Zeitverläufe


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)

if 'speed_kmh' in df.columns:
    axes[0].plot(df['elapsed_minutes'], df['speed_kmh'], color='tab:blue')
    axes[0].set_ylabel('km/h')
    axes[0].set_title('Geschwindigkeit')

if 'heart_rate' in df.columns:
    axes[1].plot(df['elapsed_minutes'], df['heart_rate'], color='tab:red')
    axes[1].set_ylabel('bpm')
    axes[1].set_title('Herzfrequenz')

if 'power' in df.columns:
    axes[2].plot(df['elapsed_minutes'], df['power'], color='tab:green')
    axes[2].set_ylabel('Watt')
    axes[2].set_title('Leistung')

axes[2].set_xlabel('Zeit (min)')
plt.tight_layout()
plt.show()


In [ ]:
if 'altitude_m' in df.columns:
    plt.figure(figsize=(15, 4))
    plt.plot(df['elapsed_minutes'], df['altitude_m'], color='saddlebrown')
    plt.title('Höhenprofil')
    plt.xlabel('Zeit (min)')
    plt.ylabel('Höhe (m)')
    plt.tight_layout()
    plt.show()
else:
    print('Keine Höhendaten vorhanden.')


## Verteilungen und Zusammenhänge


In [ ]:
if 'heart_rate' in df.columns:
    plt.figure(figsize=(10, 4))
    sns.histplot(df['heart_rate'].dropna(), bins=25, kde=True, color='tab:red')
    plt.title('Verteilung Herzfrequenz')
    plt.xlabel('bpm')
    plt.tight_layout()
    plt.show()

    heart_rate_zones(df, hr_max=190)
else:
    print('Keine Herzfrequenzdaten vorhanden.')


In [ ]:
corr = correlation_matrix(df, ['speed_kmh', 'heart_rate', 'power', 'cadence', 'altitude_m'])
if not corr.empty:
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
    plt.title('Korrelation ausgewählter Kennzahlen')
    plt.tight_layout()
    plt.show()
else:
    print('Nicht genügend Daten für eine Korrelationsmatrix vorhanden.')


## Bestleistungen


In [ ]:
best_power = rolling_best_efforts(df, column='power', windows=(5, 30, 60, 300, 1200))
if not best_power.empty:
    display(best_power)

    plt.figure(figsize=(10, 4))
    plt.plot(best_power['Fenster (s)'], best_power['Bestwert'], marker='o')
    plt.xscale('log')
    plt.title('Bestleistungen Leistung')
    plt.xlabel('Fenster (Sekunden, logarithmisch)')
    plt.ylabel('Watt')
    plt.tight_layout()
    plt.show()
else:
    print('Keine Leistungsdaten für Bestleistungen vorhanden.')


## Nächste Schritte

Mögliche Erweiterungen:

- Herzfrequenz- und Leistungszonen an deine individuellen Schwellen anpassen
- mehrere FIT-Dateien vergleichen
- automatische Exportfunktion für Berichte ergänzen
- Kartenansicht mit GPS-Daten einbauen
